<style>
div.output_scroll {
    height: auto !important;
    max-height: none !important;
}
</style>

# <font color='#3585CD'>Rappel de l'objectif</font>

Identifier les véhicules qui émettent le plus de CO2 est important pour identifier les caractéristiques techniques qui jouent un rôle dans la pollution. Prédire à l’avance cette pollution permet de prévenir dans le cas de l’apparition de nouveaux types de véhicules (nouvelles séries de voitures par exemple).

# <font color='#3585CD'>Importation des librairies</font>

In [75]:
import warnings
warnings.filterwarnings('ignore')
warnings.warn('DelftStack')
warnings.warn('Do not show this message')

import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

import seaborn as sns

import plotly.figure_factory as ff
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.stats import gaussian_kde

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.linear_model import LinearRegression, SGDRegressor, ElasticNet, Lasso, Ridge

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.tree import DecisionTreeRegressor

from xgboost import XGBRegressor

from sklearn.model_selection import learning_curve

import time

import joblib

# <font color='#3585CD'>Chargement des données</font>

In [76]:
# répertoire du dataset final (nettoyé)
dossier_fichiers = "datasets/Dataset_final/"
# répertoire d'enregistrement des modèles
dossier_modeles = "models/"

In [77]:
dataset_path = dossier_fichiers + "datas_nettoyees_model_FR.csv"
df = pd.read_csv(dataset_path)
df.head(10)

,Marque,Modèle,Masse à vide,CO2,Carburant,Cylindrée moteur,Puissance moteur,Consommation carburant
0,JEEP,RENEGADE,1845.0,49,hybride,1332.0,96.0,2.0
1,JEEP,WRANGLER,1883.0,244,essence,1995.0,200.0,10.8
2,JEEP,WRANGLER,1972.0,259,essence,1995.0,200.0,11.5
3,JEEP,WRANGLER,1972.0,258,essence,1995.0,200.0,11.5
4,JEEP,WRANGLER UNLIMITED,2348.0,79,hybride,1995.0,200.0,3.5
5,JEEP,WRANGLER UNLIMITED,2409.0,94,hybride,1995.0,200.0,4.1
6,HONDA,HR-V,1452.0,122,essence,1498.0,79.0,5.4
7,JEEP,RENEGADE,1395.0,145,essence,999.0,88.0,6.4
8,JEEP,RENEGADE,1395.0,146,essence,999.0,88.0,6.4
9,JEEP,RENEGADE,1395.0,149,essence,1332.0,110.0,6.6


# <font color='#3585CD'>Fonctions qui seront utilisées dans ce notebook</font>

In [78]:
def afficher_courbes_apprentissage(model, nom_modele):
    train_sizes, train_scores, val_scores = learning_curve(
    model, X, y, cv=5, scoring="neg_mean_absolute_error", train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
)
    # Moyenne et écart-type des scores
    train_mean = -np.mean(train_scores, axis=1)  # On remet en positif (MAE est négatif)
    train_std = np.std(train_scores, axis=1)
    val_mean = -np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    # Tracer les courbes
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_mean, 'o-', color="blue", label="Erreur d'entraînement")
    plt.plot(train_sizes, val_mean, 'o-', color="red", label="Erreur de validation")
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="blue")
    plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color="red")

    plt.xlabel("Nombre d'exemples d'entraînement")
    plt.ylabel("Erreur Absolue Moyenne (MAE)")
    plt.title("Courbe d'Apprentissage")
    plt.legend()
    plt.show()

def affichage_erreurs(y_test, y_pred, modele):
    # Calcul des résidus
    residuals = y_test - y_pred
    # Création d'une figure avec deux sous-graphiques côte à côte
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Graphique des résidus (Scatterplot)
    sns.scatterplot(x=y_pred, y=residuals, alpha=0.5, ax=axes[0])
    axes[0].axhline(y=0, color="red", linestyle="--")
    axes[0].set_xlabel("Valeurs prédites")
    axes[0].set_ylabel("Résidus")
    axes[0].set_title(f"Graphique des résidus modèle {modele}")
    axes[0].grid()

    # Histogramme des erreurs
    sns.histplot(residuals, bins=30, kde=True, ax=axes[1])
    axes[1].axvline(x=0, color="red", linestyle="--")
    axes[1].set_xlabel("Erreur (résidu)")
    axes[1].set_ylabel("Fréquence")
    axes[1].set_title(f"Histogramme des erreurs {modele}")
    axes[1].grid()

    # Affichage
    plt.tight_layout()
    plt.show()

def affichage_valeurs_reelles_vs_predites(y_test, y_pred, modele):
    plt.figure(figsize=(8, 5))
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color="red", linestyle="--")
    plt.xlabel("Valeurs réelles")
    plt.ylabel("Valeurs prédites")
    plt.title(f"Réelles vs Prédites {modele}")
    plt.grid()
    plt.show()

def affichage_QQ_plot_residus(y_test, y_pred, modele):
    residuals = y_test - y_pred
    # Configuration du style Seaborn
    sns.set(style="whitegrid")

    # Création du QQ-plot
    fig, ax = plt.subplots(figsize=(8, 5))
    stats.probplot(residuals, dist="norm", plot=ax)

    # Ajout du titre
    ax.set_title(f"QQ-Plot des résidus {modele}")

    # Affichage du graphique
    plt.show()

def afficher_courbes_apprentissage_plotly(model, nom_modele, cv=3, train_sizes=np.linspace(0.1, 1.0, 10)):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=cv, scoring="neg_mean_absolute_error",
        train_sizes=train_sizes, n_jobs=-1
    )

    train_mean = -np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    val_mean = -np.mean(val_scores, axis=1)
    val_std = np.std(val_scores, axis=1)

    fig = go.Figure()

    # Courbe d'entraînement
    fig.add_trace(go.Scatter(
        x=train_sizes, y=train_mean,
        mode='lines+markers',
        name="Erreur d'entraînement",
        line=dict(color='blue'),
    ))

    # # Bande d'écart-type entraînement
    # fig.add_trace(go.Scatter(
    #     x=np.concatenate([train_sizes, train_sizes[::-1]]),
    #     y=np.concatenate([train_mean - train_std, (train_mean + train_std)[::-1]]),
    #     fill='toself',
    #     fillcolor='rgba(0, 0, 255, 0.1)',
    #     line=dict(color='rgba(255,255,255,0)'),
    #     hoverinfo="skip",
    #     showlegend=False
    # ))

    # Courbe de validation
    fig.add_trace(go.Scatter(
        x=train_sizes, y=val_mean,
        mode='lines+markers',
        name="Erreur de validation",
        line=dict(color='red'),
    ))

    # # Bande d'écart-type validation
    # fig.add_trace(go.Scatter(
    #     x=np.concatenate([train_sizes, train_sizes[::-1]]),
    #     y=np.concatenate([val_mean - val_std, (val_mean + val_std)[::-1]]),
    #     fill='toself',
    #     fillcolor='rgba(255, 0, 0, 0.1)',
    #     line=dict(color='rgba(255,255,255,0)'),
    #     hoverinfo="skip",
    #     showlegend=False
    # ))

    fig.update_layout(
        title=f"Courbe d'Apprentissage - {nom_modele}",
        xaxis_title="Nombre d'exemples d'entraînement",
        yaxis_title="Erreur Absolue Moyenne (MAE)",
        legend=dict(
            x=1.05,
            y=1,
            xanchor='left',
            yanchor='top'
        ),
        template="plotly_white"
    )

    fig.show()

def affichage_erreurs_plotly(y_test, y_pred, modele):
    residuals = y_test - y_pred

    # Création d'une figure avec deux sous-graphiques
    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        f"Graphique des résidus modèle {modele}",
        f"Histogramme des erreurs {modele}"
    ))

    # Scatterplot des résidus
    fig.add_trace(
        go.Scatter(
            x=y_pred,
            y=residuals,
            mode='markers',
            marker=dict(opacity=0.5),
            name="Résidus"
        ),
        row=1, col=1
    )
    fig.add_hline(y=0, line=dict(color="red", dash="dash"), row=1, col=1)

    # Histogramme des résidus
    fig.add_trace(
        go.Histogram(
            x=residuals,
            nbinsx=30,
            name="Erreurs",
            marker=dict(color="blue"),
            opacity=0.7
        ),
        row=1, col=2
    )
    fig.add_vline(x=0, line=dict(color="red", dash="dash"), row=1, col=2)

    fig.update_layout(
        title_text=f"Analyse des erreurs - {modele}",
        width=1000,
        height=400,
        showlegend=False
    )
    fig.update_xaxes(title_text="Valeurs prédites", row=1, col=1)
    fig.update_yaxes(title_text="Résidus", row=1, col=1)
    fig.update_xaxes(title_text="Erreur (résidu)", row=1, col=2)
    fig.update_yaxes(title_text="Fréquence", row=1, col=2)

    fig.show()

def affichage_valeurs_reelles_vs_predites_plotly(y_test, y_pred, modele):
    df = pd.DataFrame({"Réelles": y_test, "Prédites": y_pred})

    fig = px.scatter(
        df,
        x="Réelles",
        y="Prédites",
        opacity=0.5,
        title=f"Valeurs Réelles vs Prédites - {modele}",
        labels={"Réelles": "Valeurs réelles", "Prédites": "Valeurs prédites"}
    )

    # Ajout de la ligne d'identité
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    fig.add_trace(
        go.Scatter(
            x=[min_val, max_val],
            y=[min_val, max_val],
            mode="lines",
            line=dict(color="red", dash="dash"),
            name="Ligne idéale"
        )
    )

    fig.update_layout(
        width=1000,
        height=600,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True)
    )

    fig.show()

def affichage_residus_plotly(y_test, y_pred, modele):
    residuals = y_test - y_pred

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=y_pred,
            y=residuals,
            mode='markers',
            marker=dict(opacity=0.5),
            name="Résidus"
        )
    )

    fig.add_hline(y=0, line=dict(color="red", dash="dash"))

    fig.update_layout(
        title=f"Graphique des résidus - {modele}",
        xaxis_title="Valeurs prédites",
        yaxis_title="Résidus",
        width=600,
        height=400,
        showlegend=False,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True)
    )

    fig.show()

def affichage_residus_plotly(y_test, y_pred, modele):
    residuals = y_test - y_pred

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=y_pred,
            y=residuals,
            mode='markers',
            marker=dict(opacity=0.5),
            name="Résidus"
        )
    )

    fig.add_hline(y=0, line=dict(color="red", dash="dash"))

    fig.update_layout(
        title=f"Graphique des résidus - {modele}",
        xaxis_title="Valeurs prédites",
        yaxis_title="Résidus",
        width=1000,
        height=600,
        showlegend=False,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True)
    )

    fig.show()

def affichage_histogramme_erreurs_plotly(y_test, y_pred, modele):
    residuals = y_test - y_pred

    # Calcul de la densité KDE
    kde = gaussian_kde(residuals)
    x_kde = np.linspace(residuals.min(), residuals.max(), 500)
    y_kde = kde(x_kde)

    fig = go.Figure()

    # Histogramme des résidus
    fig.add_trace(
        go.Histogram(
            x=residuals,
            nbinsx=30,
            marker=dict(color="blue"),
            opacity=0.6,
            name="Erreurs",
            histnorm='probability density'
        )
    )

    # Courbe KDE
    fig.add_trace(
        go.Scatter(
            x=x_kde,
            y=y_kde,
            mode="lines",
            line=dict(color="orange", width=2),
            name="Densité KDE"
        )
    )

    # Ligne verticale à 0
    fig.add_vline(x=0, line=dict(color="red", dash="dash"))

    fig.update_layout(
        title=f"Histogramme des erreurs avec KDE - {modele}",
        xaxis_title="Erreur (résidu)",
        yaxis_title="Densité de probabilité",
        width=1000,
        height=600,
        showlegend=True,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True)
    )

    fig.show()

def affichage_histogramme_erreurs_plotly(y_test, y_pred, modele):
    residuals = y_test - y_pred

    # Calcul de la densité KDE
    kde = gaussian_kde(residuals)
    x_kde = np.linspace(residuals.min(), residuals.max(), 500)
    y_kde = kde(x_kde)

    fig = go.Figure()

    # Histogramme des résidus avec bordure discrète et alpha
    fig.add_trace(
        go.Histogram(
            x=residuals,
            nbinsx=50,
            marker=dict(
                color="blue",
                line=dict(color="lightgray", width=1)  # Bordure discrète
            ),
            opacity=0.5,  # Transparence douce
            name="Histogramme des erreurs",
            histnorm='probability density'
        )
    )

    # Courbe KDE
    fig.add_trace(
        go.Scatter(
            x=x_kde,
            y=y_kde,
            mode="lines",
            line=dict(color="orange", width=2),
            name="Courbe KDE"
        )
    )

    # Ligne verticale à 0
    fig.add_vline(x=0, line=dict(color="red", dash="dash"))

    fig.update_layout(
        title=f"Histogramme des erreurs avec KDE - {modele}",
        xaxis_title="Erreur (résidu)",
        yaxis_title="Densité de probabilité",
        width=1000,
        height=600,
        showlegend=True,
        xaxis=dict(showgrid=True),
        yaxis=dict(showgrid=True)
    )

    fig.show()

def plot_correlation_matrix(df):
  """
  Affiche la matrice de corrélation des variables numériques sous forme de heatmap interactive avec Plotly.

  :param df: DataFrame Pandas contenant les données
  """
  # Sélection des colonnes numériques
  num_numeric_cols = df.select_dtypes(include=['number']).columns

  # Calcul de la matrice de corrélation
  corr_matrix = df[num_numeric_cols].corr()

  # Création de la heatmap avec Plotly (labels en bas et à gauche)
  fig = ff.create_annotated_heatmap(
      z=corr_matrix.values,
      x=list(corr_matrix.columns),
      y=list(corr_matrix.index),
      colorscale="RdBu_r",
      annotation_text=corr_matrix.round(2).values,
      showscale=True
  )

  # Ajustement de la disposition
  fig.update_layout(
      title="Matrice de corrélation des variables numériques",
      height=600, width=800,
      xaxis=dict(side="bottom", tickangle=-45),
      yaxis=dict(side="left")
  )

  # Affichage
  fig.show()

# <font color='#3585CD'>Informations sur le dataset</font>

In [79]:
print("\nAperçu du dataset :")
print(df.info())


Aperçu du dataset :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23895 entries, 0 to 23894
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Marque                  23895 non-null  object 
 1   Modèle                  23895 non-null  object 
 2   Masse à vide            23895 non-null  float64
 3   CO2                     23895 non-null  int64  
 4   Carburant               23895 non-null  object 
 5   Cylindrée moteur        23895 non-null  float64
 6   Puissance moteur        23895 non-null  float64
 7   Consommation carburant  23895 non-null  float64
dtypes: float64(4), int64(1), object(3)
memory usage: 1.5+ MB
None


Nous avons **23 895** lignes et **8** colonnes.

# <font color='#3585CD'>Rappel des corrélations</font>

In [80]:
plot_correlation_matrix(df)

# <font color='#3585CD'>Sélection des données : variable cible et variables explicatives</font>

Notre variable cible est le CO2.

Pour prédire celle-ci nous allons dans un premier temps sélectionner les variables explicatives suivantes : 

* Masse à vide
* Cylindrée moteur
* Puissance moteur
* Consommation carburant

In [81]:
# nous faisons une copie du dataset principal
df_copy = df.copy()
# variables explicatives
X = df_copy.drop(columns=['CO2', 'Marque', 'Modèle'], axis=1)

In [82]:
# variable cible
y = df_copy['CO2']

# <font color='#3585CD'>Pré-processeur</font>

Afin de standardiser le traitement des données, nous allons appliquer les mêmes étapes de transformation et encodage au jeu de données. Nous passerons par un pré-processeur qui encapsulera ces étapes dans un seul objet pipeline. Il pourra ainsi être rejouer pour tous les modèles que nous voudrons tester.

In [83]:
def creer_preprocesseur(X, categorical_features):
    """
    Crée un préprocesseur sklearn (ColumnTransformer) avec :
    - Standardisation des colonnes numériques
    - Encodage one-hot des colonnes catégorielles

    :param X: DataFrame d'entrée
    :param categorical_features: Liste des colonnes catégorielles    
    """
    # Colonnes numériques = toutes sauf les catégorielles
    numeric_features = X.drop(columns=categorical_features).columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

    return preprocessor


In [84]:
categorical_features = ['Carburant']
preprocessor = creer_preprocesseur(X, categorical_features)

In [ ]:
# # Colonnes à transformer
# # Colonnes catégorielles
# categorical_features = ['Carburant']
# # Colonnes numériques
# numeric_features = X.drop(columns=categorical_features).columns.tolist()

# # Préprocesseur pour le pipeline
# preprocessor = ColumnTransformer([
#     ('num', StandardScaler(), numeric_features),
#     ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
# ])

# <font color='#3585CD'>Séparation des données</font>

In [85]:
# Split des données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

In [86]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19116 entries, 7154 to 12706
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Masse à vide            19116 non-null  float64
 1   Carburant               19116 non-null  object 
 2   Cylindrée moteur        19116 non-null  float64
 3   Puissance moteur        19116 non-null  float64
 4   Consommation carburant  19116 non-null  float64
dtypes: float64(4), object(1)
memory usage: 896.1+ KB


In [87]:
X_test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4779 entries, 5776 to 23399
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Masse à vide            4779 non-null   float64
 1   Carburant               4779 non-null   object 
 2   Cylindrée moteur        4779 non-null   float64
 3   Puissance moteur        4779 non-null   float64
 4   Consommation carburant  4779 non-null   float64
dtypes: float64(4), object(1)
memory usage: 224.0+ KB


# <font color='#3585CD'>Fonction utilisée pour entrainer et optimiser un modèle</font>

## Tracking MLflow 

Si vous souhaitez **tracker les résultats** des modèles, passer la variable **tracking** à `True`.

Attention : vous devrez au préalable **démarrer le serveur MLflow** avec la commande `mlflow ui`, en veillant à être dans le même répertoire que ce notebook

In [116]:
tracking_mlflow = True

Il sera aussi peut-être nécessaire d'installer mlflow dans votre environnement

In [89]:
import sys
!{sys.executable} -m pip install mlflow

Afin de faciliter les entrainements sur les algorithmes, nous avons créé une fonction qui pourra être appelée pour chaque algorithme. Elle permet de 

* entraîner un modèle
* l'optimiser par une méthode telle que GridSearchCV ou RandomizedSearchCV
* retourner le meilleur modèle, les meilleurs hyperparamètres, les prédictions et un tableau des résultats avec des métriques (scores, durée d'éxécution ...)
* tracker les résultats dans MLflow

In [72]:

# def entrainer_et_optimiser_model(model, param_grid, model_name, preprocessor, cv=5, method='GridSearchCV', save_model=True):
#     """
#     Fonction qui entraîne et optimise un modèle de régression avec GridSearchCV,
#     retourne les performances et le temps d'exécution.
#     """

#     pipeline = Pipeline([
#         ('preprocessing', preprocessor),
#         ('regressor', model)
#     ])

#     scoring = {
#         'r2': 'r2',
#         'neg_mean_absolute_error': 'neg_mean_absolute_error'
#     }
#     if method == 'GridSearchCV':
#       search = GridSearchCV(
#           estimator=pipeline,
#           param_grid=param_grid,
#           scoring=scoring,
#           refit='r2',
#           cv=cv,
#           n_jobs=-1,
#           verbose=2
#       )
#     else:
#       search = RandomizedSearchCV(
#           estimator=pipeline,
#           param_distributions=param_grid,
#           scoring=scoring,
#           refit='r2',
#           cv=cv,
#           n_jobs=-1,
#           n_iter=5,
#           verbose=2
#       )

#     # Début du chrono
#     start_time = time.time()

#     # Entraînement
#     search.fit(X_train, y_train)

#     # Fin du chrono
#     end_time = time.time()
#     execution_time = end_time - start_time

#     best_params = search.best_params_
#     best_model = search.best_estimator_

#     mae_cv = -search.cv_results_['mean_test_neg_mean_absolute_error'].mean()

#     y_pred_train = best_model.predict(X_train)
#     r2_train = r2_score(y_train, y_pred_train)
#     mae_train = mean_absolute_error(y_train, y_pred_train)
#     rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

#     y_pred_test = best_model.predict(X_test)
#     r2_test = r2_score(y_test, y_pred_test)
#     mae_test = mean_absolute_error(y_test, y_pred_test)
#     rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

#     overfitting_ratio = r2_train - r2_test
#     underfitting_flag = r2_train < 0.80

#     # Sauvegarde du modèle
#     if save_model:
#         chemin_fichier = dossier_modeles + f"{model_name}_model.pkl"
#         joblib.dump(best_model, chemin_fichier)
#         print(f"Le meilleur modèle a été sauvegardé sous le nom : {model_name}_model.pkl")

#     print(f"Meilleurs hyperparamètres : {best_params}")
#     print(f"Temps d'exécution : {execution_time:.2f} secondes")

#     results = pd.DataFrame([{
#         'Modèle': model_name,
#         'R² moyen (CV)': search.best_score_,
#         'MAE moyen (CV)': mae_cv,
#         'RMSE train': rmse_train,
#         'MAE train': mae_train,
#         'R2 train': r2_train,
#         'RMSE test': rmse_test,
#         'MAE test': mae_test,
#         'R2 test': r2_test,
#         'Ecart RMSE / MAE test' : rmse_test - mae_test,
#         'Ratio overfitting': overfitting_ratio,
#         'Alerte Underfitting': underfitting_flag,
#         'Temps d\'exécution (s)': execution_time
#     }])

#     return best_model, best_params, y_pred_test, results


In [118]:
import mlflow
import mlflow.sklearn

def entrainer_et_optimiser_model(model, param_grid, model_name, preprocessor, cv=5, method='GridSearchCV', save_model=True, tracking=False, run_name=''):
    """
    Fonction qui entraîne et optimise un modèle de régression avec GridSearchCV ou RandomizedSearchCV,
    retourne les performances et le temps d'exécution. Peut logger les résultats dans MLflow.
    """

    pipeline = Pipeline([
        ('preprocessing', preprocessor),
        ('regressor', model)
    ])

    scoring = {
        'r2': 'r2',
        'neg_mean_absolute_error': 'neg_mean_absolute_error'
    }

    if method == 'GridSearchCV':
        search = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            scoring=scoring,
            refit='r2',
            cv=cv,
            n_jobs=-1,
            verbose=2
        )
    else:
        search = RandomizedSearchCV(
            estimator=pipeline,
            param_distributions=param_grid,
            scoring=scoring,
            refit='r2',
            cv=cv,
            n_jobs=-1,
            n_iter=5,
            verbose=2
        )

    start_time = time.time()
    search.fit(X_train, y_train)
    end_time = time.time()
    execution_time = end_time - start_time

    best_params = search.best_params_
    best_model = search.best_estimator_

    # Résultats sur train/test
    y_pred_train = best_model.predict(X_train)
    r2_train = r2_score(y_train, y_pred_train)
    mae_train = mean_absolute_error(y_train, y_pred_train)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))

    y_pred_test = best_model.predict(X_test)
    r2_test = r2_score(y_test, y_pred_test)
    mae_test = mean_absolute_error(y_test, y_pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

    overfitting_ratio = r2_train - r2_test
    underfitting_flag = r2_train < 0.80

    # Moyenne du MAE cross-val
    mae_cv = -search.cv_results_['mean_test_neg_mean_absolute_error'].mean()

    # Sauvegarde du modèle
    if save_model:
        chemin_fichier = dossier_modeles + f"{model_name}_model.pkl"
        joblib.dump(best_model, chemin_fichier)
        print(f"Le meilleur modèle a été sauvegardé sous le nom : {model_name}_model.pkl")

    print(f"Meilleurs hyperparamètres : {best_params}")
    print(f"Temps d'exécution : {execution_time:.2f} secondes")

    # Création tableau de résultats
    results = pd.DataFrame([{
        'Modèle': model_name,
        'R² moyen (CV)': search.best_score_,
        'MAE moyen (CV)': mae_cv,
        'RMSE train': rmse_train,
        'MAE train': mae_train,
        'R2 train': r2_train,
        'RMSE test': rmse_test,
        'MAE test': mae_test,
        'R2 test': r2_test,
        'Ecart RMSE / MAE test': rmse_test - mae_test,
        'Ratio overfitting': overfitting_ratio,
        'Alerte Underfitting': underfitting_flag,
        'Temps d\'exécution (s)': execution_time
    }])

    # Tracking MLflow
    # Au préalable penser à lancer MLFlow via la commande mlflow ui dans le même répertoire que le notebook
    if tracking:
        mlflow.set_experiment("Projet CO2")
        if run_name == '':
            run_name = model_name
        with mlflow.start_run(run_name=run_name) as parent_run:

            # Log du meilleur modèle
            mlflow.log_params(best_params)
            mlflow.log_metrics({
                'R2_train': r2_train,
                'MAE_train': mae_train,
                'RMSE_train': rmse_train,
                'R2_test': r2_test,
                'MAE_test': mae_test,
                'RMSE_test': rmse_test,
                'MAE_CV': mae_cv,
                'Execution_time': execution_time,
                'Overfitting_ratio': overfitting_ratio
            })
            input_example = X_train[:1]  # 1 ligne d’exemple du jeu d'entraînement
            mlflow.sklearn.log_model(
                sk_model=best_model,
                artifact_path=f"{model_name}_model",
                input_example=input_example,
                signature=mlflow.models.infer_signature(X_train, y_pred_train),
                registered_model_name=model_name
            )

            # Log CSV des résultats
            cv_results_df = pd.DataFrame(search.cv_results_)
            path_csv = f"{dossier_modeles}results/{model_name}_cv_results.csv"
            cv_results_df.to_csv(path_csv, index=False)
            mlflow.log_artifact(path_csv)

            # Log chaque itération comme une sous-run
            for i, row in cv_results_df.iterrows():
                with mlflow.start_run(run_name=f"{model_name}_test_{i}", nested=True):
                    # Log des paramètres testés
                    for param in row.index:
                        if param.startswith("param_"):
                            mlflow.log_param(param.replace("param_", ""), row[param])
                    # Log des scores
                    mlflow.log_metrics({
                        'mean_test_r2': row['mean_test_r2'],
                        'mean_test_mae': -row['mean_test_neg_mean_absolute_error'],
                        'std_test_r2': row['std_test_r2'],
                        'rank_test_r2': row['rank_test_r2']
                    })

    return best_model, best_params, y_pred_test, results


# <font color='#3585CD'>Modèle LinearRegression</font>

Le premier modèle que nous testerons est le LinearRegression

## Entrainement et optimisation

In [103]:
# Grille des hyperparamètres à tester
param_grid_lr = {
    'regressor__fit_intercept': [True, False]
}
model_name = 'LinearRegression'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_avec_consommation_carburant"

best_model_lr, best_params_lr, predictions_lr, resultats_lr = entrainer_et_optimiser_model(
    LinearRegression(), 
    param_grid_lr, 
    model_name, 
    preprocessor=preprocessor,
    run_name=run_name,
    tracking=tracking_mlflow)

resultats_lr

Fitting 5 folds for each of 2 candidates, totalling 10 fits


ValueError: 
All the 10 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\d.leloup\AppData\Local\anaconda3\Lib\site-packages\pandas\core\indexes\base.py", line 3805, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7081, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas\\_libs\\hashtable_class_helper.pxi", line 7089, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Consommation carburant'

The above exception was the

Nous constatons que le score R2 est à plus de 99%. Testons en enlevant la variable Consommation carburant qui a une très forte corrélation avec la variable cible.

In [92]:
# nous faisons une copie du dataset principal
df_sans_consommation = df.copy()
# variables explicatives
X_bis = df_sans_consommation.drop(columns=['CO2', 'Marque', 'Modèle', 'Consommation carburant'], axis=1)
# variable cible
y = df_copy['CO2']

In [93]:
categorical_features = ['Carburant']
preprocessor_bis = creer_preprocesseur(X_bis, categorical_features)

In [94]:
X_train, X_test, y_train, y_test = train_test_split(X_bis, y, test_size=0.2, random_state=24)

In [120]:
# Grille des hyperparamètres à tester
param_grid_lr = {
    'regressor__fit_intercept': [True, False]
}

model_name = 'LinearRegression'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_lr, best_params_lr, predictions_lr, resultats_lr = entrainer_et_optimiser_model(
    LinearRegression(), 
    param_grid_lr, 
    model_name, 
    preprocessor=preprocessor_bis, 
    tracking=tracking_mlflow,
    run_name=run_name)

resultats_lr

Fitting 5 folds for each of 2 candidates, totalling 10 fits
Le meilleur modèle a été sauvegardé sous le nom : LinearRegression_model.pkl
Meilleurs hyperparamètres : {'regressor__fit_intercept': True}
Temps d'exécution : 1.45 secondes


Registered model 'LinearRegression' already exists. Creating a new version of this model...
Created version '2' of model 'LinearRegression'.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,LinearRegression,0.860336,12.800338,18.126207,12.793923,0.860587,18.18486,12.705385,0.857977,5.479475,0.002611,False,1.451794


### Comment détecter un surapprentissage ?


Comparer RMSE / MAE entre train et test.

Observer l’écart entre R² train et R² test :

Si R² train ≈ R² test ➡️ OK.

Si R² train très haut et R² test plus bas ➡️ ⚠️ Surapprentissage.

Calculer un ratio d’overfitting *r2_train - r2_test* : Plus ce ratio est élevé, plus il y a un risque de surapprentissage.

### Comment détecter le sous-apprentissage ?

R² faible sur le train (< 0.80)

RMSE / MAE élevés sur le train

Si R² train ≈ R² test, mais les deux sont bas

## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeur réelle': y_test.values.ravel(),
    'Valeur prédite': predictions_lr,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)

In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_lr, 'LinearRegression')

## Courbe d'apprentissage

### Objectif

La courbe d'apprentissage permet de vérifier si le modèle apprend bien avec plus de données.
<br/><p><b>Rappel :</b> </p>
<p><b>1. Courbe idéale (Bonne généralisation)</b></p>
<P>L’erreur d’entraînement diminue progressivement.</P>
<P>L’erreur de validation suit une trajectoire similaire.</P>
<P>Les deux erreurs se stabilisent à des valeurs proches.</P>
<p><b>2. Surajustement (Overfitting)</b></p>
<P>L’erreur d’entraînement est très faible (voire 0).</P>
<P>L’erreur de validation est élevée et ne diminue pas.</P>
<P>Le modèle a appris "par cœur" et ne généralise pas bien.</P>
<p><b>3. Sous-ajustement (Underfitting)</b></p>
<P>L’erreur d’entraînement et de validation restent élevées.</P>
<P>Le modèle est trop simple et n’arrive pas à capturer la relation entre X et y.</P>
<ul>
    <li>Si l'erreur de test est très supérieure à l'erreur d'entraînement → Sur-apprentissage.</li>
<li>Si l'erreur est élevée sur les deux courbes → Sous-apprentissage.</li>
</ul>

### Courbe

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_lr, 'LinearRegression')

Modèle bien généralisant :

Pas de gros écart entre l’erreur d'entraînement et de validation.
Pas de signe fort de surapprentissage (les deux erreurs restent proches).

➤ Sous-apprentissage léger :
Vu que l'erreur de validation ne baisse pas malgré plus de données, le modèle atteint probablement ses limites de complexité.

Typique des modèles linéaires : ils ne peuvent capturer que des relations linéaires, donc si le problème est un peu plus complexe, il plafonne.

## Affichage des erreurs de prédictions

### Objectif

La **courbe des résidus** permet de vérifier si les erreurs sont aléatoires ou s'il y a une tendance dans les erreurs du modèle.
<ul>
    <li>Bonne prédiction : les points doivent être répartis aléatoirement autour de 0.
</li>
<li>Mauvaise prédiction : structure visible (courbe, tendance…).</li>
</ul>

**L'histogramme des erreurs** permet de vérifier si les erreurs suivent une distribution normale.
<ul>
    <li>Bonne prédiction : la distribution est centrée sur 0 (forme de courbe en cloche).
</li>
<li>Mauvaise prédiction : la distribution asymétrique ou biaisée.</li>
</ul>

### Graphiques

In [ ]:
affichage_residus_plotly(y_test, predictions_lr, 'LinearRegression')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_lr, 'LinearRegression')

In [ ]:
affichage_erreurs(y_test, predictions_lr, 'LinearRegression')

# Modèle Lasso

## Entrainement et optimisation

In [ ]:
# Grille des hyperparamètres à tester
param_grid_lasso = {
    'regressor__alpha': [0.0001, 0.001, 0.01, 0.1, 1, 10],
    'regressor__max_iter': [1000, 5000],
    'regressor__tol': [0.0001, 0.001],
}

model_name = 'Lasso'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_lasso, best_params_lasso, predictions_lasso, resultats_lasso = entrainer_et_optimiser_model(
    Lasso(), 
    param_grid_lasso, 
    model_name, 
    preprocessor=preprocessor_bis,
    tracking=tracking_mlflow,
    run_name=run_name)

resultats_lr

Fitting 5 folds for each of 24 candidates, totalling 120 fits
Le meilleur modèle a été sauvegardé sous le nom : Lasso_model.pkl
Meilleurs hyperparamètres : {'regressor__alpha': 0.001, 'regressor__max_iter': 1000, 'regressor__tol': 0.0001}
Temps d'exécution : 0.58 secondes


2025/04/04 09:56:29 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,LinearRegression,0.860336,12.800338,18.126207,12.793923,0.860587,18.18486,12.705385,0.857977,5.479475,0.002611,False,0.09061


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_lasso,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)

In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_lasso, 'Lasso')

## Courbe d'apprentissage

### Courbe

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_lasso, 'Lasso')

## Affichage des erreurs de prédictions

### Graphiques

In [ ]:
affichage_residus_plotly(y_test, predictions_lasso, 'Lasso')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_lasso, 'Lasso')

# Modèle Ridge

## Entrainement et optimisation

In [ ]:
# Grille des hyperparamètres à tester
param_grid_ridge = {
    'regressor__alpha': [0.1, 1, 10, 100],
}

model_name = 'Ridge'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_ridge, best_params_ridge, predictions_ridge, resultats_ridge = entrainer_et_optimiser_model(
    Ridge(), 
    param_grid_ridge, 
    model_name, 
    preprocessor=preprocessor_bis,
    tracking=tracking_mlflow,
    run_name=run_name)
resultats_ridge

Fitting 5 folds for each of 4 candidates, totalling 20 fits
Le meilleur modèle a été sauvegardé sous le nom : Ridge_model.pkl
Meilleurs hyperparamètres : {'regressor__alpha': 1}
Temps d'exécution : 0.13 secondes


2025/04/04 10:16:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,Ridge,0.860335,12.799658,18.126237,12.793238,0.860587,18.185061,12.704407,0.857974,5.480654,0.002613,False,0.133979


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_ridge,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)


In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_ridge, 'Ridge')

## Courbe d'apprentissage

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_ridge, 'Ridge')

## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_ridge, 'Ridge')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_ridge, 'Ridge')

# Modèle SGDRegressor

## Entrainement et optimisation

In [ ]:
# Grille des hyperparamètres à tester
# param_grid_sgdr = {
#     'regressor__alpha': [0.001, 0.01],
#     'regressor__penalty': ['l1', 'l2', 'elasticnet'],
#     'regressor__l1_ratio': [0.15, 0.5]
# }

model_name = 'SGDRegressor'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

param_grid_sgdr = {
    'regressor__loss': ['squared_error', 'huber'],
    'regressor__penalty': ['l2', 'l1', 'elasticnet'],
    'regressor__alpha': [0.0001, 0.001, 0.01],
    'regressor__learning_rate': ['optimal', 'invscaling'],
    'regressor__eta0': [0.01, 0.1]
}
best_model_sgdr, best_params_sgdr, predictions_sgdr, resultats_sgdr = entrainer_et_optimiser_model(
    SGDRegressor(), 
    param_grid_sgdr, 
    model_name, 
    preprocessor=preprocessor_bis,
    tracking=tracking_mlflow,
    run_name=run_name)
resultats_sgdr

Fitting 5 folds for each of 72 candidates, totalling 360 fits
Le meilleur modèle a été sauvegardé sous le nom : SGDRegressor_model.pkl
Meilleurs hyperparamètres : {'regressor__alpha': 0.01, 'regressor__eta0': 0.01, 'regressor__learning_rate': 'invscaling', 'regressor__loss': 'squared_error', 'regressor__penalty': 'l1'}
Temps d'exécution : 17.44 secondes


2025/04/04 10:18:23 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,SGDRegressor,0.860075,214780.45393,18.174877,12.920546,0.859838,18.205488,12.851801,0.857654,5.353686,0.002183,False,17.444685


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_sgdr,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)

In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_sgdr, 'SGDRegressor')

## Courbe d'apprentissage

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_sgdr, 'SGDRegressor')

L’erreur de validation chute fortement entre 5k et 20k données.

Ensuite, les gains sont faibles → le modèle semble atteindre un plateau.

## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_sgdr, 'SGDRegressor')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_sgdr, 'SGDRegressor')

# Modèle RandomForestRegressor

## Entrainement et optimisation

In [111]:
# Grille des hyperparamètres à tester
# param_grid_rfr = {
#     'regressor__n_estimators': [100, 200],   
#     'regressor__max_features': ['sqrt'],     # Garder seulement 'sqrt'
#     'regressor__max_depth': [30]             # Tester juste 30
# }

param_grid_rfr = {
    'regressor__n_estimators': [100, 200],
    'regressor__max_depth': [None, 10, 20],
    'regressor__min_samples_split': [2, 5],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__max_features': ['sqrt', 'log2'],
    'regressor__bootstrap': [True, False],
    'regressor__criterion': ['squared_error', 'absolute_error']
}

model_name = 'RandomForestRegressor'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_rfr, best_params_rfr, predictions_rfr, resultats_rfr = entrainer_et_optimiser_model(
    RandomForestRegressor(), 
    param_grid_rfr, 
    model_name, 
    preprocessor=preprocessor_bis, 
    cv=2, 
    method='RandomizedSearchCV',
    tracking=tracking_mlflow,
    run_name=run_name)

resultats_rfr

Fitting 2 folds for each of 5 candidates, totalling 10 fits
Le meilleur modèle a été sauvegardé sous le nom : RandomForestRegressor_model.pkl
Meilleurs hyperparamètres : {'regressor__n_estimators': 100, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 20, 'regressor__criterion': 'absolute_error', 'regressor__bootstrap': False}
Temps d'exécution : 258.84 secondes


2025/04/04 10:24:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,RandomForestRegressor,0.982567,4.463307,3.915405,2.627164,0.993495,5.569318,3.57764,0.986679,1.991679,0.006816,False,258.835659


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_rfr,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)


In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_rfr, 'RandomForestRegressor')

## Importance des variables (courbe d'apprentissage trop longue)

In [ ]:
# Récupérer le préprocessing depuis le meilleur modèle
preprocessor = best_model_rfr.named_steps['preprocessing']

# Noms des features après transformation
feature_names = preprocessor.get_feature_names_out()

# # Vérifier que les tailles correspondent
# print(len(feature_names), len(best_model_rfr.named_steps['regressor'].feature_importances_))


In [ ]:
importances = best_model_rfr.named_steps['regressor'].feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by="Importance", ascending=False).head(10)


In [ ]:
fig = px.bar(
    importance_df,
    x='Importance',
    y='Feature',
    orientation='h',
    title="Importance des variables"
)
fig.show()


## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_rfr, 'RandomForestRegressor')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_rfr, 'RandomForestRegressor')

In [ ]:
# Calcul des résidus (erreurs)
residus = y_test - predictions_rfr

# Ajout dans un DataFrame pour analyse
import pandas as pd
resultats_erreurs = pd.DataFrame({
    'Réel': y_test,
    'Prédiction': predictions_rfr,
    'Résidu': residus
})


In [ ]:
# Tri décroissant des erreurs absolues
resultats_erreurs['Erreur_absolue'] = resultats_erreurs['Résidu'].abs()
top_erreurs = resultats_erreurs.sort_values(by='Erreur_absolue', ascending=False).head(10)
top_erreurs

In [ ]:
# Index des plus grosses erreurs
index_erreurs = top_erreurs.index

# Récupération des lignes concernées dans X_test
X_test.loc[index_erreurs]


# Modèle DecisionTreeRegressor

## Entrainement et optimisation

In [112]:
# Grille des hyperparamètres à tester
param_grid_dtr = {
    'regressor__max_depth': [None, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__min_samples_split': [2, 5],
    'regressor__criterion': ['squared_error', 'absolute_error']
}
# param_grid_dtr = {
#     'regressor__max_depth': [10, 20, 30],
#     'regressor__min_samples_leaf': [5, 10, 20]
# }

model_name = 'DecisionTreeRegressor'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_dtr, best_params_dtr, predictions_dtr, resultats_dtr = entrainer_et_optimiser_model(
    DecisionTreeRegressor(random_state=24), 
    param_grid_dtr, 
    model_name, 
    preprocessor=preprocessor_bis, 
    cv=3,
    tracking=tracking_mlflow,
    run_name=run_name)

resultats_dtr

Fitting 3 folds for each of 36 candidates, totalling 108 fits
Le meilleur modèle a été sauvegardé sous le nom : DecisionTreeRegressor_model.pkl
Meilleurs hyperparamètres : {'regressor__criterion': 'squared_error', 'regressor__max_depth': None, 'regressor__min_samples_leaf': 1, 'regressor__min_samples_split': 2}
Temps d'exécution : 15.97 secondes


2025/04/04 10:25:16 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,DecisionTreeRegressor,0.983564,6.8237,3.670346,2.612957,0.994284,6.291549,3.537474,0.983,2.754075,0.011284,False,15.971207


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_dtr,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)


In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_dtr, 'DecisionTreeRegressor')

## Courbe d'apprentissage

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_dtr, 'DecisionTreeRegressor')

## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_dtr, 'DecisionTreeRegressor')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_dtr, 'DecisionTreeRegressor')

# Modèle GradientBoostingRegressor

## Entrainement et optimisation

In [113]:
# Grille des hyperparamètres à tester
# param_grid_gb = {
#     'regressor__n_estimators': [100],
#     'regressor__learning_rate': [0.1],
#     'regressor__max_depth': [3],
# }
param_grid_gb = {
    'regressor__n_estimators': [100, 200],    # Nombre d'arbres
    'regressor__learning_rate': [0.05, 0.1], # Taux d'apprentissage
    'regressor__max_depth': [3, 5],          # Profondeur maximale des arbres
    'regressor__min_samples_leaf': [1, 5]    # Nombre minimal d'échantillons par feuille
}

model_name = 'GradientBoostingRegressor'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_gb, best_params_gb, predictions_gb, resultats_gb = entrainer_et_optimiser_model(
    GradientBoostingRegressor(random_state=42),
    param_grid_gb,
    model_name,
    preprocessor=preprocessor_bis,
    tracking=tracking_mlflow,
    run_name=run_name
)

# Affichage des résultats
resultats_gb

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Le meilleur modèle a été sauvegardé sous le nom : GradientBoostingRegressor_model.pkl
Meilleurs hyperparamètres : {'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__min_samples_leaf': 1, 'regressor__n_estimators': 200}
Temps d'exécution : 15.96 secondes


2025/04/04 10:26:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,GradientBoostingRegressor,0.978174,6.992128,6.287595,4.847629,0.983225,7.154134,5.189392,0.978019,1.964742,0.005207,False,15.962296


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_gb,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)


In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_gb, 'GradientBoostingRegressor')

## Courbe d'apprentissage

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_gb, 'GradientBoostingRegressor')

## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_gb, 'GradientBoostingRegressor')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_gb, 'GradientBoostingRegressor')

# Modèle XGBoostRegressor

## Entrainement et optimisation

In [114]:
# Grille des hyperparamètres à tester
param_grid_xgb = {
    'regressor__n_estimators': [100, 200],    # Nombre d'arbres
    'regressor__learning_rate': [0.05, 0.1], # Taux d'apprentissage
    'regressor__max_depth': [3, 5],          # Profondeur maximale des arbres
    'regressor__min_samples_leaf': [1, 5]    # Nombre minimal d'échantillons par feuille
}

# param_grid_xgb = {
#     'regressor__n_estimators': [100, 300],
#     'regressor__learning_rate': [0.05, 0.1],
#     'regressor__max_depth': [3, 5],
#     'regressor__subsample': [0.8, 1.0]  # Pour contrôler le surapprentissage
# }

model_name = 'XGBoostRegressor'
# Si le tracking MLFlow est activé, le nom suivant sera donné au Run
run_name = f"{model_name}_sans_consommation_carburant"

best_model_xgb, best_params_xgb, predictions_xgb, resultats_xgb = entrainer_et_optimiser_model(
    XGBRegressor(random_state=42, n_jobs=-1, verbosity=1),
    param_grid_xgb,
    model_name,
    preprocessor=preprocessor_bis,
    tracking=tracking_mlflow,
    run_name=run_name
)

# Affichage des résultats
resultats_xgb

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Le meilleur modèle a été sauvegardé sous le nom : XGBoostRegressor_model.pkl
Meilleurs hyperparamètres : {'regressor__learning_rate': 0.1, 'regressor__max_depth': 5, 'regressor__min_samples_leaf': 1, 'regressor__n_estimators': 200}
Temps d'exécution : 1.92 secondes


2025/04/04 10:27:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


,Modèle,R² moyen (CV),MAE moyen (CV),RMSE train,MAE train,R2 train,RMSE test,MAE test,R2 test,Ecart RMSE / MAE test,Ratio overfitting,Alerte Underfitting,Temps d'exécution (s)
0,XGBoostRegressor,0.976889,7.104306,6.661756,5.10074,0.981169,7.45905,5.383698,0.976105,2.075352,0.005064,False,1.922318


## Comparaison des valeurs réelles et prédites

In [ ]:
# On récupère l'index des valeurs de y_test
index_test = y_test.index

# On extrait les colonnes 'Marque' et 'Modèle' depuis df_copy via l'index de test
marques = df_copy.loc[index_test, 'Marque'].values
modeles = df_copy.loc[index_test, 'Modèle'].values

# On crée le DataFrame de résultats complet
df_resultats = pd.DataFrame({
    'Valeurs réelles': y_test.values.ravel(),
    'Valeurs prédites': predictions_xgb,
    'Marque': marques,
    'Modèle': modeles
})

df_resultats.head(10)


In [ ]:
affichage_valeurs_reelles_vs_predites_plotly(y_test, predictions_xgb, 'XGBoostRegressor')

## Courbe d'apprentissage

In [ ]:
afficher_courbes_apprentissage_plotly(best_model_xgb, 'XGBoostRegressor')

## Affichage des erreurs de prédictions

In [ ]:
affichage_residus_plotly(y_test, predictions_xgb, 'XGBoostRegressor')

In [ ]:
affichage_histogramme_erreurs_plotly(y_test, predictions_xgb, 'XGBoostRegressor')

# Conclusion

In [ ]:
# Concaténation des résultats
resultats_concat = pd.concat([resultats_lr, resultats_lasso, resultats_ridge, resultats_sgdr, resultats_rfr, resultats_dtr, resultats_gb, resultats_xgb], ignore_index=True)
resultats_concat

In [ ]:
# Tri du DataFrame par R² test décroissant
resultats_concat_sorted = resultats_concat.sort_values(by='R2 test', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=resultats_concat_sorted['Modèle'],
    y=resultats_concat_sorted['R2 test'],
    name='R2 Test',
    text=resultats_concat_sorted['R2 test'].round(6),
    textposition='outside'
))

fig.update_layout(
    title='Comparaison des scores R2 test par modèle',
    xaxis_title='Modèle',
    yaxis_title='R2 Test',
    height=600,
    width=1000
)

fig.show()


In [ ]:
resultats_concat_sorted = resultats_concat.sort_values(by='RMSE test', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=resultats_concat_sorted['Modèle'],
    y=resultats_concat_sorted['RMSE test'],
    name='RMSE test',
    text=resultats_concat_sorted['RMSE test'].round(6),
    textposition='outside'
))

fig.update_layout(
    title='Comparaison des RMSE test par modèle',
    xaxis_title='Modèle',
    yaxis_title='RMSE Test',
    height=600,
    width=1000
)

fig.show()

In [ ]:
resultats_concat_sorted = resultats_concat.sort_values(by='MAE test', ascending=False)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=resultats_concat_sorted['Modèle'],
    y=resultats_concat_sorted['MAE test'],
    name='MAE test',
    text=resultats_concat_sorted['MAE test'].round(6),
    textposition='outside'
))

fig.update_layout(
    title='Comparaison des MAE test par modèle',
    xaxis_title='Modèle',
    yaxis_title='MAE Test',
    height=600,
    width=1000
)

fig.show()

In [ ]:
resultats_concat = resultats_concat.sort_values(by="Ecart RMSE / MAE test", ascending=False)

# Création du graphique avec dégradé de couleur
fig = px.bar(
    resultats_concat,
    x='Modèle',
    y="Ecart RMSE / MAE test",
    text=resultats_concat["Ecart RMSE / MAE test"].round(2),
    color="Ecart RMSE / MAE test",
    color_continuous_scale='RdYlGn_r',
    title="Ecart RMSE / MAE test"
)

# Personnalisation
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Modèle',
    yaxis_title='Ecart RMSE / MAE test',
    height=600,
    width=1000,
    coloraxis_colorbar=dict(
        title="Ecart RMSE / MAE test"
    )
)

fig.show()


Plus l’écart RMSE - MAE est grand plus il y a des grosses erreurs isolées dans ce modèle (sensibilité de RMSE aux outliers).

Si l’écart est faible, ça veut dire que les erreurs sont assez homogènes.

In [ ]:
resultats_concat = resultats_concat.sort_values(by="Temps d'exécution (s)", ascending=False)

# Création du graphique avec dégradé de couleur
fig = px.bar(
    resultats_concat,
    x='Modèle',
    y="Temps d'exécution (s)",
    text=resultats_concat["Temps d'exécution (s)"].round(2),
    color="Temps d'exécution (s)",
    color_continuous_scale='RdYlGn_r',
    title="Temps d'exécution des modèles"
)

# Personnalisation
fig.update_traces(textposition='outside')
fig.update_layout(
    xaxis_title='Modèle',
    yaxis_title='Temps d\'exécution (s)',
    height=600,
    width=1000,
    coloraxis_colorbar=dict(
        title="Temps (s)"
    )
)

fig.show()


Comment choisir ?

🔹 1️⃣ Meilleur R² test (qualité globale) :
✅ 

🔹 2️⃣ Plus petite MAE test (erreur moyenne) :
✅ 

🔹 3️⃣ Plus petit RMSE test (erreur quadratique) :
✅ 

🔹 4️⃣ Temps de calcul :

Gestion des outliers

Moi je partirais sur XGBoost :

Meilleur score global :

A le R² test le plus élevé (0.999762) (ça veut dire qu'il explique 99.9762 % de la variance de la cible).

A une MAE test et RMSE test très basses, proches du meilleur.

A un écart train/test faible (peu de surapprentissage).

Et reste raisonnable en temps de calcul.

Stable.

Bon temps d'exécution.

Moins sensible au surapprentissage que l'arbre seul.

# Exemple de prédiction


```
# Path to the saved model
model_path = dossier_modeles + f"LinearRegression_model.pkl"
loaded_model = joblib.load(model_path)

# Example New Data Input
new_data = pd.DataFrame({
    "Masse à vide": [1845],
    "Carburant": ["hybride"],
    "Cylindrée moteur": [1332],
    "Puissance moteur": [96],
    "Consommation carburant": [2]
})

# Predict CO2 Emissions
prediction = loaded_model.predict(new_data)
print("\n🚀 CO2 Emissions Prediction for New Input Data:")
print(f"Predicted CO2 Emissions: {prediction[0]:.2f} g/km")
```

